# Tutorial: rediscover 6J5W in EMD-0683

[6J5W](https://www.rcsb.org/structure/6J5W) is an Arabidopsis thaliana ZAR1–RKS1
complex fitted to the 3.7 Å map [EMD-0683](https://www.ebi.ac.uk/emdb/EMD-0683).
We treat ZAR1 (chain A, UniProt Q38834) as the unknown protein and RKS1 (chain
B) as its known partner. We remove the RKS1 density from the map, fit candidate
structures into the remaining density with PowerFit, and refine the fits against
RKS1 with HADDOCK3.

Protein-detective is intended for low-resolution maps (about 5–8 Å). EMD-0683
is a high-resolution map (<4.5 Å), used here because its deposited model lets us
check the results.

![EMD-0683 density with the fitted 6J5W model](EMD-0683.png)

EMD-0683 with the fitted 6J5W model: ZAR1 (orange, bottom) is the protein to
rediscover, RKS1 (green, top) is the known partner.

<!--
Screenshot from the
[EMDB 3D viewer](https://www.ebi.ac.uk/emdb/EMD-0683?tab=3dview)
with reduced volume opacity.
-->

## 1. Requirements

- [protein-detective](index.md) on your `PATH` (Python 3.13 or later)
- `curl` and `gzip`
- [ChimeraX](https://www.rbvi.ucsf.edu/chimerax/), run as Flatpak by default;
  set the `CHIMERAX` environment variable (for example `CHIMERAX=chimerax`) for
  a native install
- a GPU for PowerFit

All files go into the session directory `tutorial-6j5w/`, next to this notebook.

## 2. Search for candidates

Search reviewed Arabidopsis proteins annotated with ADP binding
([GO:0043531](https://www.ebi.ac.uk/QuickGO/term/GO:0043531)), with canonical
sequences of 800–900 residues. AlphaFold candidates are skipped to keep the
tutorial short.

In [1]:
%%bash
set -euo pipefail

protein-detective candidates search tutorial-6j5w \
    --taxon-id 3702 \
    --reviewed \
    --molecular-function-go GO:0043531 \
    --min-sequence-length 800 \
    --max-sequence-length 900 \
    --limit-uniprot 100 \
    --alphafold.limit 0 \
    --pdbe.limit 50 \
    --pdbe.min-residues 750 \
    --pdbe.max-residues 900 \
    --pdbe.top-resolution-per-uniprot-accession 2

Searching for UniProt accessions


Found 27 UniProt accessions, written to tutorial-6j5w/uniprot.txt
Finding PDB 

entries for 27 uniprot accessions


Before filtering found 10 PDB entries for 3 uniprot accessions.


After filtering on chain length (750, 900) remained 7[

...

                                              ac…  


Searching for quality scores for PDBe entries 100% ━━━

 4/4  [ 0:… < 0:0… , ?   ]
                             

                                              ac…  


Written quality scores for 4 PDBe entries to tutorial-6j5w/pdbe-quality.json


In [2]:
import pandas as pd

pd.read_csv("tutorial-6j5w/pdbe.csv")

,uniprot_accession,pdb_id,method,resolution,uniprot_chains,chain,chain_length
0,Q38834,6J5T,Electron_Microscopy,3.4,C/F/G/L/O=1-852,C,852
1,Q38834,6J5W,Electron_Microscopy,3.7,A=1-852,A,852
2,Q9FKZ1,8YL6,Electron_Microscopy,3.1,C=1-809,C,809
3,Q9LZ25,9JBN,Electron_Microscopy,2.3,C=1-811,C,811


Check that 6J5W was found:

In [3]:
import csv
from pathlib import Path

session = Path("tutorial-6j5w")
with (session / "pdbe.csv").open() as stream:
    ids = {row["pdb_id"].upper() for row in csv.DictReader(stream)}
assert "6J5W" in ids, ids
assert len(ids) > 1, ids
print("Search candidates:", sorted(ids))

Search candidates: ['6J5T', '6J5W', '8YL6', '9JBN']


## 3. Retrieve and filter the structures

In [4]:
%%bash
set -euo pipefail

protein-detective candidates retrieve tutorial-6j5w

protein-detective candidates filter tutorial-6j5w \
    --min-residues 750 \
    --max-residues 900 \
    --min-geometry-quality 50 \
    --top-uniprot-cluster 2 \
    --scheduler-address sequential

ls tutorial-6j5w/combined_output

Retrieving 4 PDBe entries


Retrieved 4 PDBe entries, written to tutorial-6j5w/downloads/pdbe


Statistics written to tutorial-6j5w/pdbe_stats.csv


Retrieving 0 AlphaFold entries with formats {'cif'}


Retrieved 0 AlphaFold files and 0 summaries, written to 
tutorial-6j[0

...

1;36m4, passed: 2, discarded: 2

Structures with resolution: 4,

 passed: 2, discarded: 2

Structures with geometry quality: 2, pass

ed: 0, discarded: 2
Statistics written to tutorial-6j5w/combined_st

ats.csv


6j5t_updated_C2A.cif.gz
6j5w_updated_A2A.cif.gz


In [5]:
pd.read_csv("tutorial-6j5w/combined_stats.csv")

,input_file,structure_id,uniprot_accession,resolution,high_confidence_residues_count,total_residue_count,method,is_alphafold,uniprot_start,uniprot_end,sequence_identity,chain_length,geometry_quality,passed,output_file,reason
0,combined_input/8yl6_updated_C2A.cif.gz,8YL6,Q9FKZ1,3.1,NaN,397,EM,False,1,809,1.0,397,71.82,False,NaN,"Chain length 397 not in range [750, 900]"
1,combined_input/9jbn_updated_C2A.cif.gz,9JBN,Q9LZ25,2.3,NaN,423,EM,False,1,811,1.0,423,61.40,False,NaN,"Chain length 423 not in range [750, 900]"
2,combined_input/6j5t_updated_C2A.cif.gz,6J5T,Q38834,3.4,NaN,808,EM,False,1,852,1.0,808,NaN,True,combined_output/6j5t_updated_C2A.cif.gz,NaN
3,combined_input/6j5w_updated_A2A.cif.gz,6J5W,Q38834,3.7,NaN,816,EM,False,1,852,1.0,816,NaN,True,combined_output/6j5w_updated_A2A.cif.gz,NaN


Two structures of ZAR1 pass: 6J5T (`6j5t_updated_C2A.cif.gz`) and 6J5W
(`6j5w_updated_A2A.cif.gz`). The other candidates have too few residues.

## 4. Download the experimental map and reference complex

In [6]:
%%bash
set -euo pipefail

curl -fsSL --retry 3 \
    https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-0683/map/emd_0683.map.gz \
    -o tutorial-6j5w/emd_0683.map.gz

gzip -dk tutorial-6j5w/emd_0683.map.gz

curl -fsSL --retry 3 \
    https://www.ebi.ac.uk/pdbe/entry-files/download/6j5w_updated.cif.gz \
    -o tutorial-6j5w/6j5w_updated.cif.gz

## 5. Remove the known RKS1 density from the map

Simulate a 3.7 Å map of RKS1 and zero the experimental map inside it, as in
[the EM preparation notebook](em_prepare.ipynb).

In [7]:
%%bash
set -euo pipefail

gzip -dk tutorial-6j5w/6j5w_updated.cif.gz

cat > tutorial-6j5w/mask-rks1.cxc <<'EOF'
open emd_0683.map
open 6j5w_updated.cif
delete #2/A
molmap #2 3.7
volume mask #1 surfaces #3 invertMask true
save emd_0683-without-rks1.mrc models #4
exit
EOF

(cd tutorial-6j5w &&
    ${CHIMERAX:-flatpak run edu.ucsf.rbvi.ChimeraX} --nogui --exit --script mask-rks1.cxc)

INFO:
Executing: runscript mask-rks1.cxc


INFO:
Executing: open emd_0683.map


INFO:
Opened emd_0683.map as #1, grid size 160,160,160, pixel 1.09, shown at level 0.0364, step 1, v

alues float32


INFO:
Executing: open 6j5w_updated.cif


...

INFO:
Opened emd_0683.map masked as #4, grid size 160,160,160, pixel 1.09, shown at step 1, values f

loat32


INFO:
Executing: save emd_0683-without-rks1.mrc models #4


STATUS:
Wrote file emd_0683-without-rks1.mrc


INFO:
Executing: exit
STATUS:
Exiting ...


Check that the density at RKS1 is gone and at ZAR1 is kept:

In [8]:
from pathlib import Path

import gemmi
import numpy as np

session = Path("tutorial-6j5w")
structure = gemmi.read_structure(str(session / "6j5w_updated.cif"))
grid = gemmi.read_ccp4_map(str(session / "emd_0683-without-rks1.mrc"), setup=True).grid
mean_density = {}
for chain in structure[0]:
    values = [grid.interpolate_value(atom.pos) for residue in chain for atom in residue if atom.name == "CA"]
    mean_density[chain.name] = float(np.mean(values))
    print(chain.name, "mean density at CA atoms:", round(mean_density[chain.name], 4))
assert mean_density["B"] == 0, mean_density
assert mean_density["A"] > 0, mean_density

B mean density at CA atoms: 0.0
A mean density at CA atoms: 0.0818


## 6. Fit both candidates into the masked EMD-0683 map

Fitting both candidates takes about 40 seconds on an NVIDIA GeForce RTX 3050.

In [9]:
%%bash
set -euo pipefail

protein-detective powerfit run \
    tutorial-6j5w/emd_0683-without-rks1.mrc 3.7 \
    tutorial-6j5w \
    --angle 10 \
    --scheduler-address sequential \
    --powerfit-run-id emd0683

protein-detective powerfit list-runs tutorial-6j5w \
    --output tutorial-6j5w/powerfit-runs.csv

protein-detective powerfit report tutorial-6j5w \
    --powerfit-run-id emd0683 \
    --top 1 \
    --output tutorial-6j5w/powerfit-report.csv


Running PowerFit sequentially:   0%|          | 0/2 [00:00<?, ?structure/s]


Running PowerFit sequentially:  50%|█████     | 1/2 [00:18<00:18, 18.40s/structure]


Running PowerFit sequentially: 100%|██████████| 2/2 [00:36<00:00, 18.15s/struct

ure]


Running PowerFit sequentially: 100%|██████████| 2/2 [00:36<00:00, 18.19s/struct

ure]


PowerFit run completed with ID: emd0683. Use this ID for reporting or fitting 
models.


In [10]:
pd.read_csv("tutorial-6j5w/powerfit-report.csv")

,powerfit_run_id,structure,rank,cc,fishz,relz,translation,rotation,template_file,uniprot_accessions,structure_id,is_alphafold
0,emd0683,6j5w_updated_A2A.cif.gz,1,0.310,0.320,44.290,87.28:90.553:97.099,1.0:0.0:0.0:0.0:1.0:0.0:0.0:0.0:1.0,tutorial-6j5w/combined_output/6j5w_updated_A2A...,Q38834,6J5W,False
1,emd0683,6j5t_updated_C2A.cif.gz,1,0.066,0.066,9.082,85.098:92.735:105.827,-0.087:0.568:0.818:-0.273:0.776:-0.568:-0.958:...,tutorial-6j5w/combined_output/6j5t_updated_C2A...,Q38834,6J5T,False


The report has the best solution per candidate; higher `cc` is better. 6J5W
scores 0.31 in its deposited pose, 6J5T only 0.066.

Write the fitted models:

In [11]:
%%bash
set -euo pipefail

protein-detective powerfit fit-models tutorial-6j5w \
    --powerfit-run-id emd0683 \
    --top 1 \
    --output tutorial-6j5w/fitted-models.csv


Writing fitted model PDB files:   0%|          | 0/2 [00:00<?, ?it/s]


Writing fitted model PDB files: 100%|██████████| 2/2 [00:00<00:00, 27.98it/s]


## 7. Refine against the known RKS1 partner

Use 6J5W without chain A as the fixed RKS1 partner. The small sampling settings
keep the run to a few minutes.

In [12]:
%%bash
set -euo pipefail

protein-detective refine run \
    tutorial-6j5w \
    tutorial-6j5w/6j5w_updated.cif.gz \
    --remove-fixed-chains A \
    --powerfit-run-id emd0683 \
    --refine-run-id emd0683-small \
    --rigidbody-sampling 10 \
    --top-clusters 1 \
    --top-models 1 \
    --water-refinement-sampling-factor 1 \
    --water-refinement-solvent none \
    --ncores "$(nproc)" \
    --scheduler-address sequential

protein-detective refine list-runs tutorial-6j5w \
    --output tutorial-6j5w/refine-runs.csv

protein-detective refine report tutorial-6j5w \
    --refine-run-id emd0683-small \
    --top 1 \
    --output tutorial-6j5w/refined.csv


Refined structures:   0%|          | 0/2 [00:00<?, ?file/s]

[2026-10-14 13:26:45,597 libclust WARNING] Reordering by cluster rank and model rank


[2026-10-14 13:27:32,512 libclust WARNING] Reordering by cluster rank and model rank



Refined structures:  50%|█████     | 1/2 [03:24<03:24, 204.64s/file]

[2026-10-14 13:29:47,576 libclust WARNING] Reordering by cluster rank and model rank


[2026-10-14 13:30:33,062 libclust WARNING] Reordering by cluster rank and model rank



Refined structures: 100%|██████████| 2/2 [06:24<00:00, 190.16s/file]


Refined structures: 100%|██████████| 2/2 [06:24<00:00, 192.33s/file]


Refinement run completed with ID: emd0683-small


In [13]:
display(pd.read_csv("tutorial-6j5w/refine-runs.csv"))
pd.read_csv("tutorial-6j5w/refined.csv")

,refine_run_id,run_dir,fixed_structure,options
0,emd0683-small,tutorial-6j5w/refine/emd0683-small,/home/verhoes/git/protein-detective/protein-de...,--refine-run-id emd0683-small --rigidbody-samp...


,refine_run_id,powerfit_run_id,structure,fitted_rank,uniprot_accessions,cluster_id,cluster_rank,population,under_eval,cluster_score,cluster_score_std,model_rank,unrefined_score,model_score,model_path,fitted_model,refine_run_dir
0,emd0683-small,emd0683,6j5t_updated_C2A.cif.gz,1,Q38834,1,1,1,True,-96.508,0.0,1,-30.430,-96.508,tutorial-6j5w/refine/emd0683-small/emd0683/6j5...,tutorial-6j5w/powerfit/emd0683/6j5t_updated_C2...,tutorial-6j5w/refine/emd0683-small/emd0683/6j5...
1,emd0683-small,emd0683,6j5w_updated_A2A.cif.gz,1,Q38834,1,1,1,True,-85.504,0.0,1,-258.006,-85.504,tutorial-6j5w/refine/emd0683-small/emd0683/6j5...,tutorial-6j5w/powerfit/emd0683/6j5w_updated_A2...,tutorial-6j5w/refine/emd0683-small/emd0683/6j5...


Lower HADDOCK scores are better. Before refinement 6J5W scores −258.0 and 6J5T
−30.4; after refinement 6J5W scores −85.5 and 6J5T −96.5. Ten rigid-body samples
are too few to keep the native pose, so increase `--rigidbody-sampling` for real
use.

## 8. Visualize fitted models

<!--
TODO: add a `protein-detective visualize` CLI command that shows the fitted
(PowerFit) and refined (HADDOCK3) structures together with the density map,
so this chapter can use a single command instead of the Python script below.
-->

Write an HTML viewer of the fitted models and the masked map:

In [14]:
from pathlib import Path

from protein_detective.visualization import (
    load_fitted_models,
    show_fitted_models_and_density,
)

root = Path("tutorial-6j5w")
models = load_fitted_models(
    fitted_models_csv=root / "fitted-models.csv",
    report_csv=root / "powerfit-report.csv",
    runs_csv=root / "powerfit-runs.csv",
)
html = show_fitted_models_and_density(models, renderer="html")
(root / "fitted-models.html").write_text(html)

23255764

Open `tutorial-6j5w/fitted-models.html` in a browser. See
[the visualization notebook](visualization.ipynb) for more examples.

## 9. Build the metadata database and check the results

In [15]:
%%bash
set -euo pipefail

protein-detective meta tutorial-6j5w

tutorial-6j5w/meta.duckdb has been generated.
See https://www.bo

nvinlab.org/protein-detective/meta.html for example queries.
Use `duckdb --readonly tutorial-[1

;36m6j5w/meta.duckdb` to query it.


In [16]:
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

root = Path("tutorial-6j5w")
report = pd.read_csv(root / "powerfit-report.csv")
assert "6J5W" in set(report["structure_id"].str.upper())
assert len(report) == 2, report
assert np.isfinite(report["cc"]).all()
fitted = pd.read_csv(root / "fitted-models.csv")
assert len(fitted) == 2
assert all(Path(path).is_file() for path in fitted["fitted_model_file"])
refined = pd.read_csv(root / "refined.csv")
assert not refined.empty
assert np.isfinite(refined[["model_score", "unrefined_score"]]).all().all()
assert all(Path(path).is_file() for path in refined["model_path"])
with duckdb.connect(str(root / "meta.duckdb"), read_only=True) as db:
    print(db.sql("SHOW TABLES"))
    print(
        db.sql("""
        SELECT structure_id, max(cc) AS best_cc
        FROM solutions
        GROUP BY structure_id
        ORDER BY best_cc DESC
    """)
    )
print("Pipeline outputs verified.")

┌──────────────────────────┐
│           name           │
│         varchar          │
├──────────────────────────┤
│ alphafold                │
│ alphafold_retrieve_stats │
│ combined_stats           │
│ fittable_structures      │
│ fitted_models            │
│ merge_structure_files    │
│ pdbe                     │
│ pdbe_retrieve_stats      │
│ refinements_capri_clt    │
│ refinements_capri_ss     │
│ refinements_io           │
│ refinements_unrefined    │
│ rocrate_create_actions   │
│ rocrate_inodes           │
│ rocrate_nodes            │
│ rocrate_objects          │
│ rocrate_results          │
│ solutions                │
│ structure_files          │
│ uniprot                  │
│ uniprots_verified_stats  │
├──────────────────────────┤
│         21 rows          │
└──────────────────────────┘

┌──────────────┬─────────┐
│ structure_id │ best_cc │
│   varchar    │  float  │
├──────────────┼─────────┤
│ 6J5W         │    0.31 │
│ 6J5T         │   0.066 │
└──────────────┴─────────

See [the metadata notebook](meta.ipynb) for queries on `meta.duckdb`.